In [ ]:
!pip install -q \
    "pyarrow==18.1.0" \
    "datasets==3.6.0" \
    "transformers==4.55.4" \
    "accelerate==1.10.1" \
    "peft==0.17.1" \
    "bitsandbytes==0.46.1" \
    "trl==0.21.0" \
    "evaluate==0.4.3" \
    "rouge_score" \
    "nltk" \
    "bert-score" \
    "sentence-transformers" \
    "textstat" \
    "scikit-learn"


In [ ]:
from huggingface_hub import login

login()

from google.colab import drive

drive.mount("/content/drive")

import os
import re
import json
import math
import numpy as np
import pandas as pd
import torch

from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/Llama_5K_Radiology_Project"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "llama32_1b_qlora_5000"
)

ADAPTER_DIR = os.path.join(
    BASE_DIR,
    "llama32_1b_qlora_5000_adapter"
)

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Output:", OUTPUT_DIR)
print("Adapter:", ADAPTER_DIR)
print("Results:", RESULTS_DIR)


In [ ]:
from datasets import load_dataset

SEED = 42

dataset = load_dataset(
    "BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track"
)

shuffled = dataset["train"].shuffle(seed=SEED)

training_pool = shuffled.select(range(10000))

train_data = training_pool.select(range(5000))

validation_data = shuffled.select(range(10000, 10100))

evaluation_dataset = shuffled.select(range(10100, 10600))

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(evaluation_dataset))

assert len(train_data) == 5000
assert len(validation_data) == 100
assert len(evaluation_dataset) == 500

split_df = pd.DataFrame({
    "split": (
        ["train_5k"] * 5000 +
        ["validation"] * 100 +
        ["test"] * 500
    ),
    "index_in_shuffled_dataset": (
        list(range(5000)) +
        list(range(10000, 10100)) +
        list(range(10100, 10600))
    )
})

split_path = os.path.join(
    RESULTS_DIR,
    "5k_fixed_split.csv"
)

split_df.to_csv(split_path, index=False)

print("Saved:", split_path)


In [ ]:
def build_text(example):
    return {
        "text": (
            "### Radiology Report:\n"
            + str(example["radiology_report"])
            + "\n\n"
            "### Layman Summary:\n"
            + str(example["layman_report"])
        )
    }

train_data = train_data.map(build_text)
validation_data = validation_data.map(build_text)

print(train_data[0]["text"][:1000])


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto"
)

model.config.use_cache = False

print("Model loaded.")


In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    fp16=True,
    bf16=False,

    max_length=384,

    logging_steps=50,

    save_strategy="steps",
    save_steps=750,
    save_total_limit=5,

    eval_strategy="steps",
    eval_steps=750,

    lr_scheduler_type="cosine",

    seed=42,

    gradient_checkpointing=True,

    report_to="none"
)


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=train_data,
    eval_dataset=validation_data,

    peft_config=peft_config,

    processing_class=tokenizer
)

print("Starting 5K training...")

trainer.train()

print("Training completed.")


In [ ]:
print("\nCHECKPOINTS SAVED:\n")

for item in sorted(os.listdir(OUTPUT_DIR)):
    if item.startswith("checkpoint-"):
        print(item)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Adapter saved:")
print(ADAPTER_DIR)

history_path = os.path.join(
    RESULTS_DIR,
    "llama32_1b_5k_training_history.csv"
)

history_df = pd.DataFrame(trainer.state.log_history)

history_df.to_csv(
    history_path,
    index=False
)

print("Saved:", history_path)


In [ ]:
from peft import PeftModel

torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

model.eval()

print("5K fine-tuned model loaded.")


In [ ]:
predictions = []
references = []
reports = []

for i, sample in enumerate(evaluation_dataset):

    report = str(sample["radiology_report"])
    reference = str(sample["layman_report"])

    prompt = f"""
You are a medical communication assistant.

Convert the following radiology report into a simple,
patient-friendly explanation.

Do not add medical findings that are not present.
Do not add diagnoses, symptoms, treatment, causes,
recommendations, or prognosis.

Radiology Report:
{report}

Patient-friendly summary:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    reports.append(report)
    references.append(reference)
    predictions.append(generated)

    if (i + 1) % 50 == 0:
        print(f"Generated {i + 1}/500")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Llama_5K_Radiology_Project"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "llama32_1b_qlora_5000"
)

ADAPTER_DIR = os.path.join(
    BASE_DIR,
    "llama32_1b_qlora_5000_adapter"
)

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results"
)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:
predictions = []
references = []
reports = []

for i, sample in enumerate(evaluation_dataset):

    report = str(sample["radiology_report"])
    reference = str(sample["layman_report"])

    prompt = f"""
You are a medical communication assistant.

Convert the following radiology report into a simple,
patient-friendly explanation.

Do not add medical findings that are not present.
Do not add diagnoses, symptoms, treatment, causes,
recommendations, or prognosis.

Radiology Report:
{report}

Patient-friendly summary:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    reports.append(report)
    references.append(reference)
    predictions.append(generated)

    if (i + 1) % 50 == 0:
        print(f"Generated {i + 1}/500")


In [ ]:
results_df = pd.DataFrame({
    "radiology_report": reports,
    "reference_layman_summary": references,
    "generated_summary": predictions
})

generated_path = os.path.join(
    RESULTS_DIR,
    "finetuned_llama_5k_500_generated_summaries.csv"
)

results_df.to_csv(
    generated_path,
    index=False
)

print("Saved:", generated_path)


In [ ]:
import nltk

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("punkt_tab")

from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score

from rouge_score import rouge_scorer

references_tokenized = [
    [ref.lower().split()]
    for ref in references
]

predictions_tokenized = [
    pred.lower().split()
    for pred in predictions
]

bleu = corpus_bleu(
    references_tokenized,
    predictions_tokenized
)

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for ref, pred in zip(references, predictions):

    score = scorer.score(ref, pred)

    rouge1_scores.append(score["rouge1"].fmeasure)
    rouge2_scores.append(score["rouge2"].fmeasure)
    rougeL_scores.append(score["rougeL"].fmeasure)

rouge1 = np.mean(rouge1_scores)
rouge2 = np.mean(rouge2_scores)
rougeL = np.mean(rougeL_scores)

meteor_scores = []

for ref, pred in zip(references, predictions):

    meteor_scores.append(
        meteor_score(
            [ref.lower().split()],
            pred.lower().split()
        )
    )

meteor = np.mean(meteor_scores)

print("BLEU:", bleu)
print("ROUGE-1:", rouge1)
print("ROUGE-2:", rouge2)
print("ROUGE-L:", rougeL)
print("METEOR:", meteor)


In [ ]:
from bert_score import score as bertscore

P, R, F1 = bertscore(
    predictions,
    references,
    model_type="roberta-large",
    lang="en",
    verbose=True
)

bert_precision = P.mean().item()
bert_recall = R.mean().item()
bert_f1 = F1.mean().item()

print("BERTScore Precision:", bert_precision)
print("BERTScore Recall:", bert_recall)
print("BERTScore F1:", bert_f1)


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

semantic_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)

reference_embeddings = semantic_model.encode(
    references,
    batch_size=16,
    show_progress_bar=True
)

prediction_embeddings = semantic_model.encode(
    predictions,
    batch_size=16,
    show_progress_bar=True
)

semantic_scores = [
    cosine_similarity(
        [p],
        [r]
    )[0][0]
    for p, r in zip(
        prediction_embeddings,
        reference_embeddings
    )
]

semantic_similarity = np.mean(semantic_scores)

print("Semantic Similarity:", semantic_similarity)


In [ ]:
import textstat

fkgl_scores = []
dale_scores = []
cli_scores = []

for text in predictions:

    fkgl_scores.append(
        textstat.flesch_kincaid_grade(text)
    )

    dale_scores.append(
        textstat.dale_chall_readability_score(text)
    )

    cli_scores.append(
        textstat.coleman_liau_index(text)
    )

fkgl = np.mean(fkgl_scores)
dale_chall = np.mean(dale_scores)
coleman_liau = np.mean(cli_scores)

print("FKGL:", fkgl)
print("Dale-Chall:", dale_chall)
print("Coleman-Liau:", coleman_liau)


In [ ]:
from transformers import pipeline

nli_classifier = pipeline(
    "text-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=0 if torch.cuda.is_available() else -1
)

nli_labels = []

for report, generated in zip(
    reports,
    predictions
):

    result = nli_classifier({
        "text": report,
        "text_pair": generated
    })

    if isinstance(result, list):
        label = result[0]["label"].lower()
    else:
        label = result["label"].lower()

    nli_labels.append(label)

hallucinated_samples = sum(
    label != "entailment"
    for label in nli_labels
)

hallucination_percentage = (
    hallucinated_samples / len(nli_labels)
) * 100

print(
    "Hallucinated Samples:",
    hallucinated_samples
)

print(
    "Hallucination Percentage:",
    hallucination_percentage
)
